# 15.1 Collaborative filtering

Recommendations can be made from people who behaved alike, even when we know almost nothing about the items themselves.

This lesson starts from the user-item interaction matrix: rows are users, columns are items, and missing entries are ratings we want to infer. Cosine similarity turns overlapping behavior into neighbor weights, but those weights are only trustworthy when overlap is real. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: masked neighbor evidence

For a sparse matrix $R$, user-user collaborative filtering predicts
$$\hat r_{ui}=rac{\sum_{v\in N(u)}s(u,v)r_{vi}}{\sum_{v\in N(u)}s(u,v)},\qquad s(u,v)=rac{R_u\cdot R_v}{\|R_u\|_2\|R_v\|_2}.$$
The mask is the method: zeros mean unknown, so similarities only use items both users rated.

In [ ]:

def masked_cosine(a, b, min_overlap=1):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = (a > 0.0) & (b > 0.0)
    overlap = int(np.sum(mask))
    if overlap < min_overlap:
        return 0.0, overlap
    return cosine(a[mask], b[mask]), overlap


def user_user_cf(R, user, item, min_overlap=1, centered=False):
    R = np.asarray(R, dtype=float)
    weights = []
    values = []
    overlaps = []
    target = R[user]
    observed_target = target > 0.0
    target_mean = np.mean(target[observed_target])
    for other in range(R.shape[0]):
        if other == user:
            continue
        if R[other, item] <= 0.0:
            continue
        compare = R[other]
        if centered:
            other_mean = np.mean(compare[compare > 0.0])
            target_vec = np.where(observed_target, target - target_mean, 0.0)
            other_vec = np.where(compare > 0.0, compare - other_mean, 0.0)
            sim, overlap = masked_cosine(target_vec, other_vec, min_overlap=min_overlap)
            value = R[other, item] - other_mean
        else:
            sim, overlap = masked_cosine(target, compare, min_overlap=min_overlap)
            value = R[other, item]
        if sim > 0.0:
            weights.append(sim)
            values.append(value)
            overlaps.append(overlap)
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    if len(weights) == 0:
        fallback = np.mean(R[R[:, item] > 0.0, item])
        return float(fallback), weights, overlaps
    prediction = float(np.dot(weights, values) / np.sum(weights))
    if centered:
        prediction = float(target_mean + prediction)
    return prediction, weights, overlaps


def item_item_cf(R, user, item):
    R = np.asarray(R, dtype=float)
    weights = []
    values = []
    for other_item in range(R.shape[1]):
        if other_item == item:
            continue
        if R[user, other_item] <= 0.0:
            continue
        sim, overlap = masked_cosine(R[:, item], R[:, other_item])
        if sim > 0.0:
            weights.append(sim)
            values.append(R[user, other_item])
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    return float(np.dot(weights, values) / np.sum(weights)), weights


## Check the lesson numbers

In the lesson toy, Ava's missing item uses similarities $13/27.495=0.473$ and $9/33.045=0.272$, giving $\hat r_{0,2}=4.635$. Item-item evidence changes the neighborhood axis and lowers the estimate to $3.351$.

In [ ]:

R_lesson = np.array([
    [5.0, 4.0, 0.0, 1.0],
    [4.0, 5.0, 0.0, 1.0],
    [1.0, 1.0, 5.0, 4.0],
    [0.0, 1.0, 4.0, 5.0],
])

pred_user, weights_user, overlaps_user = user_user_cf(R_lesson, 0, 2)
pred_item, weights_item = item_item_cf(R_lesson, 0, 2)

assert np.round(weights_user[0], 3) == 0.473
assert np.round(weights_user[1], 3) == 0.272
assert np.round(pred_user, 3) == 4.635
assert np.round(pred_item, 3) == 3.351

print("user-user prediction", round(pred_user, 3))
print("item-item prediction", round(pred_item, 3))
print("overlap counts", overlaps_user)


## The dataset ladder

F14 has no shared ladder here, so we build it inline: D1 tiny rating matrix, D2 small preferences, D3 noisy sparse ties, D4 inline MovieLens-like sample, and D5 long-tail cold-start.

In [ ]:

rungs = rating_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for rung in rungs:
    matrix = rung["ratings"].copy()
    true_values = []
    predicted_values = []
    for user, item, true_rating in rung["holdout"]:
        matrix[user, item] = 0.0
        pred, weights, overlaps = user_user_cf(matrix, user, item, min_overlap=1, centered=True)
        true_values.append(true_rating)
        predicted_values.append(pred)
    score = rmse(true_values, predicted_values)
    rows.append({"name": rung["name"], "rmse": score, "predictions": predicted_values, "truth": true_values})

for row in rows:
    print(f"{row['name']}: RMSE={row['rmse']:.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for row in rows:
    axes[0].scatter(row["truth"], row["predictions"], label=row["name"])
axes[0].plot([1, 5], [1, 5], color="black", linewidth=1)
axes[0].set_title("Predicted vs true ratings")
axes[0].set_xlabel("true")
axes[0].set_ylabel("predicted")
axes[0].legend(fontsize=7)
axes[1].plot([row["name"].split()[0] for row in rows], [row["rmse"] for row in rows], marker="o")
axes[1].set_title("RMSE vs sparsity rung")
axes[1].set_ylabel("RMSE")
fig.tight_layout()


## Pitfall on D5: missing-as-zero and popularity masquerading as personalization

D5 has head items and cold items. The wrong version treats every blank as a zero rating, creating fake negative evidence and pushing everyone toward the same popular head.

In [ ]:

d5 = rating_ladder()[-1]
R5 = d5["ratings"].copy()
user = 10
item = 0
wrong_sim = cosine(R5[user], R5[0])
fixed_sim, fixed_overlap = masked_cosine(R5[user], R5[0], min_overlap=1)
wrong_score = float(np.mean(R5[:, item]))
fixed_score, fixed_weights, fixed_overlaps = user_user_cf(R5, user, item, min_overlap=1, centered=True)

print("wrong full-vector cosine", round(wrong_sim, 3))
print("fixed masked cosine", round(fixed_sim, 3), "overlap", fixed_overlap)
print("wrong missing-as-zero item mean", round(wrong_score, 3))
print("fixed masked centered prediction", round(fixed_score, 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `RMSE` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Increase `min_overlap` on D5 and explain the RMSE tradeoff.
3. Compare user-user and item-item predictions for the same held-out cell.